In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model")

schema = schema.post_index()

In [ ]:
asym_params = np.load(DATA_PATH / "fit_full_asym_ising_no_structure.npz")["params"][
    8:
].reshape((8, 8))

_sym_params = np.load(DATA_PATH / "fit_full_sym_ising_no_structure.npz")["params"][8:]
sym_params = np.zeros_like(asym_params)
sym_params[np.triu_indices(8)] = _sym_params
sym_params += np.tril(sym_params.T, k=-1)

In [ ]:
asym_pol = asym_params[5]
asym_cci = asym_params[6]
asym_ccw = asym_params[2]

In [ ]:
sym_pol = sym_params[5]
sym_cci = sym_params[6]
sym_ccw = sym_params[2]

In [ ]:
asym_pol = np.delete(asym_pol, [5, 6])
asym_cci = np.delete(asym_cci, [5, 6])
sym_pol = np.delete(sym_pol, [5, 6])
sym_cci = np.delete(sym_cci, [5, 6])

In [ ]:
asym_pol = np.delete(asym_ccw, [2, 6])
asym_cci = np.delete(asym_cci, [2, 6])
sym_pol = np.delete(sym_ccw, [2, 6])
sym_cci = np.delete(sym_cci, [2, 6])

In [ ]:
import polars as pl

In [ ]:
fig, axes = plt.subplots(
    nrows=2, figsize=(3.5, 3.25), constrained_layout=True, sharey=True, sharex=True
)


df = pl.DataFrame(
    {
        "model": ["Asymmetric"] * 12 + ["Symmetric"] * 12,
        "source": ["Politics"] * 6
        + ["Climate Impacts"] * 6
        + ["Politics"] * 6
        + ["Climate Impacts"] * 6,
        "belief": np.tile(
            [
                "CC Real",
                "CC Human",
                "CC Worry",
                "CC Others Worry",
                "Weather Worry",
                "CC Action",
            ],
            4,
        ),
        "influence": np.concat([asym_pol, asym_cci, sym_pol, sym_cci]),
    }
)

sns.barplot(
    df.filter(model="Asymmetric"), x="belief", y="influence", hue="source", ax=axes[0]
)
sns.barplot(
    df.filter(model="Symmetric"), x="belief", y="influence", hue="source", ax=axes[1]
)

axes[0].set_ylabel("Asym.", rotation=0, ha="left", labelpad=10)
axes[1].set_ylabel("Symm.", rotation=0, ha="left", labelpad=10)

axes[0].yaxis.set_label_position("right")
axes[1].yaxis.set_label_position("right")

axes[0].legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.05), frameon=False)
axes[1].get_legend().remove()

axes[0].set_ylim(0, 0.25)
axes[0].set_yticks(np.linspace(0, 0.25, 2))

fig.supylabel("Outbound influence", y=0.58)

for ax in axes:
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)
    ax.set_xlabel(None)


axes[1].set_xticks(
    np.arange(6),
    [
        "CC Real",
        "CC Human",
        "CC Worry",
        "CC Others Worry",
        "Weather Worry",
        "CC Action",
    ],
    rotation=90,
)

fig.suptitle("Absolute outbound influence", fontsize=14)

fig.savefig(
    "../reports/thesis/results/figures/compare_outbound_influence.svg",
    transparent=True,
    bbox_inches="tight",
)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3.25), constrained_layout=True)

df = pl.DataFrame(
    {
        "model": ["Asymmetric"] * 12 + ["Symmetric"] * 12,
        "source": ["Politics"] * 6
        + ["Climate Impacts"] * 6
        + ["Politics"] * 6
        + ["Climate Impacts"] * 6,
        "belief": np.tile(
            [
                "CC Real",
                "CC Human",
                "CC Worry",
                "CC Others Worry",
                "Weather Worry",
                "CC Action",
            ],
            4,
        ),
        "influence": np.concat([asym_pol, asym_cci, sym_pol, sym_cci]),
    }
)

sns.barplot(
    df.filter(model="Asymmetric"),
    x="belief",
    y="influence",
    hue="source",
    ax=ax,
    palette=["#228833", "#4477AA"],
    gap=0.05,
)

ax.set_ylabel("Outbound influence")  # , rotation=0, ha="left", labelpad=10)

# axes[0].yaxis.set_label_position("right")
# axes[1].yaxis.set_label_position("right")

ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.05), frameon=False)

ax.set_ylim(0, 0.25)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_xlabel(None)

ax.set_xticks(
    np.arange(6),
    [
        "CC Real",
        "CC Human",
        "CC Worry",
        "CC Others Worry",
        "Weather Worry",
        "CC Action",
    ],
    rotation=90,
)

fig.suptitle("Asymmetric model", fontsize=14, x=0.6)

fig.savefig(
    "../reports/thesis/results/figures/compare_outbound_influence.svg",
    transparent=True,
    bbox_inches="tight",
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(3.25, 3.25), constrained_layout=True, sharey=True, sharex=True
)

diffs_pol = sym_pol - asym_pol
diffs_cci = sym_cci - asym_cci

df = pl.DataFrame(
    {
        "source": ["Politics"] * 6 + ["Climate Impacts"] * 6,
        "belief": np.tile(
            [
                "CC Real",
                "CC Human",
                "CC Worry",
                "CC Others Worry",
                "Weather Worry",
                "CC Action",
            ],
            2,
        ),
        "influence": np.concat([diffs_pol, diffs_cci]),
    }
)


sns.barplot(
    df,
    x="belief",
    y="influence",
    hue="source",
    ax=ax,
    palette=["#228833", "#4477AA"],
    gap=0.05,
)


# axes[0].set_ylabel("Asym.", rotation=0, ha="left", labelpad=10)
# axes[1].set_ylabel("Symm.", rotation=0, ha="left", labelpad=10)

# axes[0].yaxis.set_label_position("right")
# axes[1].yaxis.set_label_position("right")

ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.05), frameon=False)
# axes[1].get_legend().remove()

# axes[0].set_ylim(0, 0.25)
# axes[0].set_yticks(np.linspace(0,0.25,2))

# fig.supylabel("Outbound influence", y=0.63)

# ax.set_ylabel(r"$J_\text{symm} - J_\text{asym}$")
ax.set_ylabel("Extra influence\nin symmetric model")

# ax.annotate(
#     "Asym. model higher",
#     xy=(-0.2, 1.0), xycoords="axes fraction",
#     xytext=(-0.2, 0.45), textcoords="axes fraction",
#     arrowprops=dict(arrowstyle="->", lw=1.5),
#     ha="center",
#     va="center",
#     rotation=90,
# )

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_xlabel(None)
fig.suptitle("Difference between models", x=0.62, fontsize=14)


ax.set_xticks(
    np.arange(6),
    [
        "CC Real",
        "CC Human",
        "CC Worry",
        "CC Others Worry",
        "Weather Worry",
        "CC Action",
    ],
    rotation=90,
)
fig.savefig(
    "../reports/thesis/results/figures/difference_outbound_influence.svg",
    transparent=True,
    bbox_inches="tight",
)